# Predict survival vs non-survival for patients with admissions due to COVID-19.

In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Gather Covid-19 related admissions and capture survived vs non-survived population

In [ ]:
# Dignosed condition
conditions = pd.read_csv('/content/drive/MyDrive/Educational/UTAustin/AI in Healthcare/10k_synthea_covid19_csv/conditions.csv')


In [ ]:
#Grab the IDs of patients that have been diagnosed with COVID-19

covid_patient_ids = conditions[conditions.CODE == 840539006].PATIENT.unique()

In [ ]:
observations = pd.read_csv('/content/drive/MyDrive/Educational/UTAustin/AI in Healthcare/10k_synthea_covid19_csv/observations.csv')


In [ ]:
#This grabs every patient with a negative SARS-CoV-2 test. This will include patients
#who tested negative up front as well as patients that tested negative after leaving the hospital
negative_covid_patient_ids = observations[(observations.CODE == '94531-1') & (observations.VALUE == 'Not detected (qualifier value)')].PATIENT.unique()

In [ ]:
patients = pd.read_csv('/content/drive/MyDrive/Educational/UTAustin/AI in Healthcare/10k_synthea_covid19_csv/patients.csv')

In [ ]:
#Grabs IDs for all patients that died in the simulation. This will be more than just COVID-19 deaths.

deceased_patients = patients[patients.DEATHDATE.notna()].Id

In [ ]:
care_plans = pd.read_csv('/content/drive/MyDrive/Educational/UTAustin/AI in Healthcare/10k_synthea_covid19_csv/careplans.csv')


In [ ]:
# Grabs IDs for patients that have completed the care plan for isolation at home.

completed_isolation_patients = care_plans[(care_plans.CODE == 736376001) & (care_plans.STOP.notna()) & (care_plans.REASONCODE == 840539006)].PATIENT

In [ ]:
# Survivors are the union of those who have completed isolation at home or have a negative SARS-CoV-2 test.
survivor_ids = np.union1d(completed_isolation_patients, negative_covid_patient_ids)

In [ ]:
encounters = pd.read_csv('/content/drive/MyDrive/Educational/UTAustin/AI in Healthcare/10k_synthea_covid19_csv/encounters.csv')


In [ ]:
# Grab IDs for patients with admission due to COVID-19

inpatient_ids = encounters[(encounters.REASONCODE == 840539006) & (encounters.CODE == 1505002)].PATIENT


In [ ]:
inpatient_ids.shape

(1867,)

In [ ]:
# The number of inpatient survivors

survived_patient_ids = np.intersect1d(inpatient_ids, survivor_ids)
survived_patient_ids.shape

(1522,)

In [ ]:
#The number of inpatient non-survivors

dead_patient_ids = np.intersect1d(inpatient_ids, deceased_patients)
dead_patient_ids.shape

(349,)

In [ ]:
print(f"{survived_patient_ids[:5]=}")
print(f"{dead_patient_ids[:5]=}")

survived_patient_ids[:5]=array(['00079a57-24a8-430f-b4f8-a1cf34f90060',
       '00093cdd-a9f0-4ad8-87e9-53534501f008',
       '002dd6c0-26c5-4bd0-b0f2-3b6f600b132f',
       '005a2319-85cd-40b6-b0bc-749cef080c7e',
       '005d9b50-f564-4c86-8f90-97b0199b015a'], dtype=object)
dead_patient_ids[:5]=array(['0008a63c-c95c-46c2-9ef3-831d68892019',
       '000e7adf-cbaa-4fad-ab2f-658c32f7d4d3',
       '0100f99a-1b5d-4a5b-a73f-559a920412e5',
       '0138b15c-2616-4be2-9755-14255878cd99',
       '0143c325-d2e1-4c0c-abeb-9ec7726c309e'], dtype=object)


# OpenAI API usage with CoT and One-Shot Prompting for Dies/Survived prediction based on conditions data

In [ ]:
from openai import OpenAI
api_key="sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx" #Enter your API Key here
openai_client = OpenAI(api_key=api_key)

In [ ]:
model="gpt-4.1"

In [ ]:
known_dead_patient_id = "0008a63c-c95c-46c2-9ef3-831d68892019"
known_survived_patient_id = "00079a57-24a8-430f-b4f8-a1cf34f90060"
# verify this patient id is in dead list
print(f"{known_dead_patient_id in dead_patient_ids=}")

print(f"{known_survived_patient_id in survived_patient_ids=}")

known_dead_patient_id in dead_patient_ids=True
known_survived_patient_id in survived_patient_ids=True


In [ ]:
# Prepare one-shot example for a known Patient ID and survival status as Dies
outcomes_dead = conditions[(conditions.PATIENT == known_dead_patient_id)].DESCRIPTION.unique()
print(f"{outcomes_dead}")
# Prepare one-shot example for a known Patient ID and survival status as Survived
outcomes_survived = conditions[(conditions.PATIENT == known_survived_patient_id)].DESCRIPTION.unique()
print(f"{outcomes_survived}")

['Chronic sinusitis (disorder)' 'Body mass index 30+ - obesity (finding)'
 'Cough (finding)' 'Dyspnea (finding)' 'Wheezing (finding)'
 'Diarrhea symptom (finding)' 'Fever (finding)' 'Loss of taste (finding)'
 'Suspected COVID-19' 'COVID-19' 'Pneumonia (disorder)'
 'Hypoxemia (disorder)' 'Respiratory distress (finding)'
 'Acute respiratory failure (disorder)'
 'Sepsis caused by virus (disorder)' 'Acute pulmonary embolism (disorder)']
['Hypertension' 'Cough (finding)' 'Muscle pain (finding)'
 'Joint pain (finding)' 'Fever (finding)' 'Loss of taste (finding)'
 'Suspected COVID-19' 'COVID-19' 'Pneumonia (disorder)'
 'Hypoxemia (disorder)' 'Respiratory distress (finding)'
 'Acute pulmonary embolism (disorder)']


In [ ]:
# Provide Chain of Thought as the two-shot example
prompt_setup = "Predict survival status in one word (Survived / Died) for the patient below. Explain step-by-step based on known risk factors."
oneshot_example_dead_prompt = "Example: Patient suffering from all the following conditions: " + ", ".join(map(str, outcomes_dead))
oneshot_example_dead_cot_response = """Reasoning:
1. High-Risk Profile: The patient has obesity and chronic sinusitis, increasing susceptibility to severe respiratory infections.
2. Severe COVID-19 Presentation: Classic and worsening symptoms—fever, cough, wheezing, dyspnea, and GI signs—indicate systemic viral impact.
3. Pulmonary Complications: The infection progresses to pneumonia, leading to hypoxemia and respiratory distress.
4. Organ Failure: Patient develops acute respiratory failure, requiring critical care; oxygen exchange is severely impaired.
5. Systemic Breakdown: Sepsis caused by virus and pulmonary embolism result in systemic inflammation and vascular collapse.
6. Outcome: Multi-organ failure ensues despite intervention

Prediction: **Died**
"""

oneshot_example_survived_prompt = "Example: Patient suffering from all the following conditions: " + ", ".join(map(str, outcomes_survived))
oneshot_example_survived_cot_response = """Reasoning:
1. Moderate Baseline Risk: The patient has hypertension, but no high-risk comorbidities like obesity or diabetes.
2. Symptomatic COVID-19: Develops typical symptoms—fever, cough, myalgia, loss of taste—suggesting active viral infection.
3. Lung Involvement: Disease progresses to pneumonia with mild-to-moderate hypoxemia and respiratory distress.
4. Critical Complication Managed: Acute pulmonary embolism occurs but is identified early and treated with anticoagulants.
5. Organ Function Preserved: No signs of respiratory failure or sepsis; patient responds to oxygen and supportive therapy.
6. Outcome: Patient stabilizes and recovers without ICU-level deterioration.

Prediction: **Survived**
"""

In [ ]:
# Now pass the prompt to make model follow chain of thought for different patient as well who Died

random_dead_patient_id = dead_patient_ids[np.random.randint(len(dead_patient_ids))]
target_patient_conditions = conditions[(conditions.PATIENT == random_dead_patient_id)].DESCRIPTION.unique()
user_prompt = prompt_setup + "\n" + oneshot_example_dead_prompt + "\n" + oneshot_example_dead_cot_response + "\n" \
  + oneshot_example_survived_prompt + "\n" + oneshot_example_survived_cot_response + "\n" \
  + "----" + "\n" + "Now predict for this patient suffering from all the following conditions: " + ", ".join(map(str, target_patient_conditions))

print(user_prompt)

Predict survival status in one word (Survived / Died) for the patient below. Explain step-by-step based on known risk factors.
Example: Patient suffering from all the following conditions: Chronic sinusitis (disorder), Body mass index 30+ - obesity (finding), Cough (finding), Dyspnea (finding), Wheezing (finding), Diarrhea symptom (finding), Fever (finding), Loss of taste (finding), Suspected COVID-19, COVID-19, Pneumonia (disorder), Hypoxemia (disorder), Respiratory distress (finding), Acute respiratory failure (disorder), Sepsis caused by virus (disorder), Acute pulmonary embolism (disorder)
Reasoning: 
1. High-Risk Profile: The patient has obesity and chronic sinusitis, increasing susceptibility to severe respiratory infections.
2. Severe COVID-19 Presentation: Classic and worsening symptoms—fever, cough, wheezing, dyspnea, and GI signs—indicate systemic viral impact.
3. Pulmonary Complications: The infection progresses to pneumonia, leading to hypoxemia and respiratory distress.
4.

In [ ]:
# Get response from model
messages = [
    {"role": "system", "content": "You are a clinical decision support assistant trained to assess COVID-19 survival likelihood based on patient conditions."},
    {"role": "user", "content": user_prompt}
]

response = openai_client.chat.completions.create(
  model=model,
  messages=messages
)

print(response.choices[0].message.content)

Reasoning:
1. Very High Baseline Risk: The patient has multiple significant comorbidities—obesity (BMI 30+), hypertension, hyperlipidemia, breast malignancy, and chronic congestive heart failure. These conditions individually and synergistically increase susceptibility to severe COVID-19 outcomes.
2. Complicated Cardiac History: Chronic congestive heart failure drastically increases mortality risk in COVID-19 due to limited cardiopulmonary reserve and vulnerability to decompensation under stress (e.g., hypoxia, infection).
3. Active/Recent Malignancy: A diagnosis of malignant neoplasm of the breast suggests either active malignancy or immunosuppression (potential chemotherapy/radiation), further reducing physiological reserves and immune function.
4. Severity of COVID-19 Course: The appearance of dyspnea, wheezing, sore throat, fatigue, myalgia, joint pain, fever, and loss of taste indicate systemic and respiratory involvement.
5. Pulmonary Complications: Disease has progressed to pneu

In [ ]:
# Now pass the prompt to make model follow chain of thought for a patient who Survived

random_survived_patient_id = survived_patient_ids[np.random.randint(len(survived_patient_ids))]
target_patient_conditions = conditions[(conditions.PATIENT == random_survived_patient_id)].DESCRIPTION.unique()
user_prompt = prompt_setup + "\n" + oneshot_example_dead_prompt + "\n" + oneshot_example_dead_cot_response + "\n" \
  + oneshot_example_survived_prompt + "\n" + oneshot_example_survived_cot_response + "\n" \
  + "----" + "\n" + "Now predict for this patient suffering from all the following conditions: " + ", ".join(map(str, target_patient_conditions))

print(f"{random_survived_patient_id=}, {user_prompt=}")
# Get response from model
messages = [
    {"role": "system", "content": "You are a clinical decision support assistant trained to assess COVID-19 survival likelihood based on patient conditions."},
    {"role": "user", "content": user_prompt}
]

response = openai_client.chat.completions.create(
  model=model,
  messages=messages
)

print(response.choices[0].message.content)

random_survived_patient_id='50f3a7a2-77df-455e-85c9-1e0632b52423', user_prompt='Predict survival status in one word (Survived / Died) for the patient below. Explain step-by-step based on known risk factors.\nExample: Patient suffering from all the following conditions: Chronic sinusitis (disorder), Body mass index 30+ - obesity (finding), Cough (finding), Dyspnea (finding), Wheezing (finding), Diarrhea symptom (finding), Fever (finding), Loss of taste (finding), Suspected COVID-19, COVID-19, Pneumonia (disorder), Hypoxemia (disorder), Respiratory distress (finding), Acute respiratory failure (disorder), Sepsis caused by virus (disorder), Acute pulmonary embolism (disorder)\nReasoning: \n1. High-Risk Profile: The patient has obesity and chronic sinusitis, increasing susceptibility to severe respiratory infections.\n2. Severe COVID-19 Presentation: Classic and worsening symptoms—fever, cough, wheezing, dyspnea, and GI signs—indicate systemic viral impact.\n3. Pulmonary Complications: The

In [ ]:
N = 50

# Calculate performance based on first N dead and first N survived patients

prediction = []


for patient_id in dead_patient_ids[:N]:
  target_patient_conditions = conditions[(conditions.PATIENT == patient_id)].DESCRIPTION.unique()
  user_prompt = prompt_setup + "\n" + oneshot_example_dead_prompt + "\n" + oneshot_example_dead_cot_response + "\n" \
    + oneshot_example_survived_prompt + "\n" + oneshot_example_survived_cot_response + "\n" \
    + "----" + "\n" + "Now predict for this patient suffering from all the following conditions: " \
    + ", ".join(map(str, target_patient_conditions))

  # Get response from model
  messages = [
      {"role": "system", "content": "You are a clinical decision support assistant trained to assess COVID-19 survival likelihood based on patient conditions."},
      {"role": "user", "content": user_prompt}
  ]

  response = openai_client.chat.completions.create(
    model=model,
    messages=messages
  )
  # To parse the response
  if (response.choices[0].message.content.endswith("**Died**")):
    # Accurate
    prediction.append(1)
  else:
    prediction.append(0)

In [ ]:
for patient_id in survived_patient_ids[:N]:
  target_patient_conditions = conditions[(conditions.PATIENT == patient_id)].DESCRIPTION.unique()
  user_prompt = prompt_setup + "\n" + oneshot_example_dead_prompt + "\n" + oneshot_example_dead_cot_response + "\n" \
    + oneshot_example_survived_prompt + "\n" + oneshot_example_survived_cot_response + "\n" \
    + "----" + "\n" + "Now predict for this patient suffering from all the following conditions: " + ", ".join(map(str, target_patient_conditions))

  # Get response from model
  messages = [
      {"role": "system", "content": "You are a clinical decision support assistant trained to assess COVID-19 survival likelihood based on patient conditions."},
      {"role": "user", "content": user_prompt}
  ]

  response = openai_client.chat.completions.create(
    model=model,
    messages=messages
  )
  # To parse the response
  if (response.choices[0].message.content.endswith("**Died**")):
    prediction.append(1)
  else:
    prediction.append(0)

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, RocCurveDisplay

labels = [1] * N + [0] * N # 1st N dead and last N survived
auroc = roc_auc_score(labels, prediction)
auprc = average_precision_score(labels, prediction)
print('\nAUROC:', auroc, '\nAUPRC', auprc)


AUROC: 0.73 
AUPRC 0.6493506493506493


# Use ChatGPT embedding for survival prediction

In [ ]:
test_size = 0.2

In [ ]:
# Prepare train test split
from sklearn.model_selection import train_test_split
train_dead, test_dead = train_test_split(dead_patient_ids[:N], test_size=test_size, shuffle=True)
train_survived, test_survived = train_test_split(survived_patient_ids[:N], test_size=test_size, shuffle=True)
train_data = np.concatenate((train_dead, train_survived))
test_data = np.concatenate((test_dead, test_survived))
print(f"{train_data.shape=}, {test_data.shape=}")

train_data.shape=(80,), test_data.shape=(20,)


In [ ]:
labels = [1] * len(train_dead) + [0] * len(train_survived) # 1st N*0.8 dead and last N*0.8 survived

In [ ]:
def generate_embeddings(text, model="text-embedding-3-small"):
    response = openai_client.embeddings.create(input = text, model=model)
    return response.data[0].embedding

In [ ]:
embedding = []
for patient_id in train_data:
  target_patient_conditions = conditions[(conditions.PATIENT == patient_id)].DESCRIPTION.unique()
  prompt = prompt_setup + "\n" + oneshot_example_dead_prompt + "\n" + oneshot_example_dead_cot_response + "\n" \
    + oneshot_example_survived_prompt + "\n" + oneshot_example_survived_cot_response + "\n" \
    + "----" + "\n" + "Now predict for this patient suffering from all the following conditions: " + ", ".join(map(str, target_patient_conditions))
  embedding.append(generate_embeddings(prompt))

embedding = np.array(embedding)

In [ ]:
embedding.shape

(80, 1536)

In [ ]:
# Logistic Regression from embedding to binary classification
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(embedding, labels)

LogisticRegression(max_iter=1000)

In [ ]:
test_embedding = []
for patient_id in test_data:
  target_patient_conditions = conditions[(conditions.PATIENT == patient_id)].DESCRIPTION.unique()
  prompt = prompt_setup + "\n" + oneshot_example_dead_prompt + "\n" + oneshot_example_dead_cot_response + "\n" \
    + oneshot_example_survived_prompt + "\n" + oneshot_example_survived_cot_response + "\n" \
    + "----" + "\n" + "Now predict for this patient suffering from all the following conditions: " + ", ".join(map(str, target_patient_conditions))
  test_embedding.append(generate_embeddings(prompt))

test_embedding = np.array(test_embedding)
test_labels = [1] * len(test_dead) + [0] * len(test_survived)

test_pred = model.predict_proba(test_embedding)[:,1]
auroc = roc_auc_score(test_labels, test_pred)
auprc = average_precision_score(test_labels, test_pred)
print('\nAUROC:', auroc, '\nAUPRC', auprc)


AUROC: 0.96 
AUPRC 0.9669230769230769


In [ ]:
test_embedding.shape

(20, 1536)